# Project Garuda — Threat & Weapon Detector Training Notebook
### Fine-Tuning YOLOv8 for Automated Weapon & Lethal Threat Detection (SIH26187)

This notebook trains/fine-tunes the **YOLOv8n** threat detection model for Project Garuda using a free **Nvidia T4 GPU** on Google Colab.

**Detected Threat Classes:**
- Class 0: `Gun` (Pistols, Handguns, Rifles)
- Class 1: `knife` (Knives, Daggers, Machetes)

---
### Step 1: Check GPU Acceleration
Make sure your Colab Runtime is set to **GPU**:
*Runtime > Change runtime type > Hardware accelerator: T4 GPU*

In [ ]:
!nvidia-smi

### Step 2: Install Libraries (Ultralytics, Kagglehub, HuggingFace)

In [ ]:
%pip install -q ultralytics kagglehub huggingface_hub

---
### Step 3: Fast-Track Option — Download Official Pre-Trained Weights (5 Seconds)
If you just want the official fine-tuned weights for Project Garuda right away without waiting for training, run this cell:

In [ ]:
from huggingface_hub import hf_hub_download
from google.colab import files
import shutil

print("Downloading official fine-tuned Subh775/Threat-Detection-YOLOv8n weights...")
weights_path = hf_hub_download(repo_id="Subh775/Threat-Detection-YOLOv8n", filename="weights/best.pt")
shutil.copy(weights_path, "threat_yolov8n.pt")
print("Downloaded successfully! Triggering browser download...")
files.download("threat_yolov8n.pt")

---
### Step 4: Custom Training — Download Weapon Dataset
This automatically downloads the verified **Weapon-Detection-for-YOLO** dataset directly via `kagglehub` with zero manual login required.

In [ ]:
import kagglehub
import os
import glob

# Download full dataset
dataset_dir = kagglehub.dataset_download("sultanofdata/weapon-detection-for-yolo")
print("Dataset downloaded to:", dataset_dir)

# Locate data.yaml
yaml_matches = glob.glob(f"{dataset_dir}/**/*.yaml", recursive=True)
if yaml_matches:
    data_yaml = yaml_matches[0]
    print("Found YAML configuration file:", data_yaml)
    !cat "{data_yaml}"
else:
    raise FileNotFoundError("data.yaml not found in downloaded dataset")

### Step 5: Fine-Tune YOLOv8n on GPU (~10-15 Minutes)
- Pretrained baseline: `yolov8n.pt`
- Image size: 640x640
- Batch size: 16 (optimized for T4 16GB VRAM)
- Early stopping enabled (patience=10)

In [ ]:
from ultralytics import YOLO

# Load baseline YOLOv8 nano model
model = YOLO('yolov8n.pt')

# Train on GPU
results = model.train(
    data=data_yaml,
    epochs=25,
    imgsz=640,
    batch=16,
    device=0,
    patience=10,
    save=True,
    project='garuda_runs',
    name='threat_yolov8n',
    exist_ok=True
)

print("Training Complete!")

### Step 6: Validate Precision & Recall

In [ ]:
trained_model = YOLO('garuda_runs/threat_yolov8n/weights/best.pt')
metrics = trained_model.val()
print(f"mAP50-95: {metrics.box.map:.4f}")
print(f"mAP50: {metrics.box.map50:.4f}")

### Step 7: Download Fine-Tuned `threat_yolov8n.pt` to your Computer
This automatically downloads the fine-tuned model directly to your Downloads folder.
Then drop it into `ai_engine/models/threat_yolov8n.pt` in your Garuda workspace!

In [ ]:
from google.colab import files
import shutil

# Copy to threat_yolov8n.pt and download
shutil.copy('garuda_runs/threat_yolov8n/weights/best.pt', 'threat_yolov8n.pt')
print("Downloading fine-tuned threat_yolov8n.pt...")
files.download('threat_yolov8n.pt')